# Handwriting PD score = Spiral model + Mender model (late fusion)

One **spiral** image + one **mender** image of the **same person** → each model gives a PD probability
→ combined with weights (from each model's cross-validation performance) → **final PD % score**.

No paired training data is needed for this. It is only used at prediction time.
> Note: the combined score is **not validated** (no paired data). Only each model's own CV results are measured.

## 0. (Run first) Check for duplicate images

In [1]:
import hashlib
from pathlib import Path
import numpy as np
from PIL import Image

EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()

def ahash(p, size=16):                      # tiny perceptual hash (catches re-saved / resized copies)
    a = np.asarray(Image.open(p).convert("L").resize((size, size)), dtype=np.float32)
    return "".join("1" if v else "0" for v in (a > a.mean()).flatten())

for root in ["Spiral", "Mender"]:
    for cls in [f"{root}Control", f"{root}Patients"]:
        files = [p for p in sorted((Path(root) / cls).rglob("*")) if p.suffix.lower() in EXTS]
        exact = len({md5(p) for p in files})
        near  = len({ahash(p) for p in files})
        print(f"{cls:15s} files: {len(files):4d} | unique (exact): {exact:4d} | unique (near-identical): {near:4d}")

SpiralControl   files:    0 | unique (exact):    0 | unique (near-identical):    0
SpiralPatients  files:    0 | unique (exact):    0 | unique (near-identical):    0
MenderControl   files:    0 | unique (exact):    0 | unique (near-identical):    0
MenderPatients  files:    0 | unique (exact):    0 | unique (near-identical):    0


## 1. Imports, settings & the two CNNs (same architecture as training)

In [2]:
import json
import torch, torch.nn as nn
import pandas as pd
from torchvision import transforms

IMG_SIZE = 160
CKPT_DIR = Path("checkpoints")
RES = {"spiral": Path("results/spiral_cv"), "mender": Path("results/mender_cv")}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WEIGHT_BY = "auc"      # "auc" (recommended) or "accuracy"

def conv_block(i, o, drop):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.MaxPool2d(2), nn.Dropout2d(drop))

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(conv_block(1, 16, 0.05), conv_block(16, 32, 0.10),
                                      conv_block(32, 64, 0.10), conv_block(64, 128, 0.15),
                                      nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(128, 64),
                                        nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(self, x):
        return self.classifier(self.features(x))

tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
                         transforms.Normalize([0.5], [0.5])])

def load_ensemble(prefix):
    models = []
    for f in range(1, 6):
        m = CNN().to(DEVICE)
        m.load_state_dict(torch.load(CKPT_DIR / f"{prefix}_cv_fold{f}.pt", map_location=DEVICE)["model"])
        m.eval(); models.append(m)
    return models

MODELS = {"spiral": load_ensemble("spiral"), "mender": load_ensemble("mender")}
print("Loaded 5 spiral + 5 mender fold models on", DEVICE)

Loaded 5 spiral + 5 mender fold models on cpu


## 2. Weights from each model's CV performance

In [3]:
perf = {}
for k, d in RES.items():
    fm = pd.read_csv(d / "fold_metrics.csv")
    perf[k] = {"auc": fm.auc.mean(), "accuracy": fm.accuracy.mean(),
               "thr": json.load(open(d / "threshold.json"))["threshold"]}

tot = sum(perf[k][WEIGHT_BY] for k in perf)
W = {k: perf[k][WEIGHT_BY] / tot for k in perf}
FUSED_THR = sum(W[k] * perf[k]["thr"] for k in perf)

for k in perf:
    print(f"{k:7s} CV AUC {perf[k]['auc']:.3f} | CV accuracy {perf[k]['accuracy']:.3f} "
          f"| threshold {perf[k]['thr']:.3f} | weight {W[k]:.3f}")
print(f"Fused decision threshold: {FUSED_THR:.3f}")

spiral  CV AUC 0.897 | CV accuracy 0.836 | threshold 0.389 | weight 0.491
mender  CV AUC 0.932 | CV accuracy 0.876 | threshold 0.401 | weight 0.509
Fused decision threshold: 0.395


## 3. Predict: one spiral + one mender image → PD %

In [5]:
def model_score(kind, image_path):
    x = tf(Image.open(image_path).convert("L")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return float(np.mean([torch.sigmoid(m(x)).item() for m in MODELS[kind]]))   # mean of 5 fold models

def predict_pd(spiral_path, mender_path):
    s, m = model_score("spiral", spiral_path), model_score("mender", mender_path)
    fused = W["spiral"] * s + W["mender"] * m
    label = "Parkinson (PD)" if fused >= FUSED_THR else "Healthy"
    print(f"Spiral model : {s*100:5.1f} %  (CV AUC {perf['spiral']['auc']:.2f}, weight {W['spiral']:.2f})")
    print(f"Mender model : {m*100:5.1f} %  (CV AUC {perf['mender']['auc']:.2f}, weight {W['mender']:.2f})")
    print(f"--> Handwriting PD score: {fused*100:.1f} %   => {label}")
    return fused

# Example (change the two paths - both images must be from the SAME person):
predict_pd("../dataset/Spiral_HandPD/SpiralPatients/0002-1.jpg", "../dataset/Meander/MenderPatients/0002-5.jpg")

Spiral model :  87.7 %  (CV AUC 0.90, weight 0.49)
Mender model :  90.1 %  (CV AUC 0.93, weight 0.51)
--> Handwriting PD score: 88.9 %   => Parkinson (PD)


np.float64(0.8889470865470879)

In [6]:
import sys
print(sys.executable)

c:\Users\kanis\AppData\Local\Programs\Python\Python314\python.exe
